In [12]:
import numpy as np
import pandas as pd
import requests
from pathlib import Path 
import yaml
import re
import json

config_path= "../configs/London.yaml"

with Path(config_path).open("r", encoding="utf-8") as f:
                config = yaml.safe_load(f)
listing_loc = pd.read_csv(Path("../data/processed/listings-loc.csv"), dtype = {"listing_id": "str"}, usecols = ["listing_id", "analysis_area"])

for date in config['snapshots']:

    calendar_path = Path(f"../data/raw/{date}/calendar.csv.gz")
    calendar = pd.read_csv(calendar_path, compression='gzip', dtype = {"listing_id": "str"})
    calendar_ghep = calendar.merge(listing_loc[["listing_id"]], on = "listing_id", how = "right")
    calendar_ghep["available"] = calendar_ghep["available"].map({"t": True, "f": False}).astype("boolean")
    #xu ly tiep calendar
    calendar_ghep['date']=pd.to_datetime(calendar_ghep['date'],format='%Y-%m-%d',errors="coerce")
    cap=config['qa']['min_nights_cap']
    calendar_ghep['flag_date'] = calendar_ghep['date'].isna()
    calendar_ghep['flag_available'] = calendar_ghep['available'].isna()
    calendar_ghep['flag_min_max_outlier'] = (calendar_ghep['minimum_nights']>cap) | (calendar_ghep['minimum_nights']<0) | (calendar_ghep['maximum_nights']<0)
    calendar_ghep['flag_min_max_odd'] = (calendar_ghep['minimum_nights']>calendar_ghep['maximum_nights'])
    calendar_ghep['flag_duplicated']=calendar_ghep.duplicated(subset=['listing_id','date'], keep='first')

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\2026-03-21\\calendar.csv.gz'